In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")
from IPython.display import display, Javascript

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

C:\Users\HDong\source\repos\BKRCastTestBed\BKR4-24-v1\scripts\summarize\calibration\notebooks\../../../../scripts\h5toDF.py:96: SyntaxWarning:

invalid escape sequence '\A'



In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Household and Population

In [3]:
def hh_and_pop(data1, data3, tag='PSRC Region'):
    # Merge data
    merge_per_hh_1 = pd.merge(data1['Person'][['pwtyp', 'psexpfac', 'pwpcl', 'pwaudist','pstyp', 'pspcl', 'psaudist', 'hhno', 'ptpass']],
                            data1['Household'][['hhtaz', 'hhparcel', 'hhno']],
                            on = 'hhno')
    merge_per_hh_3 = pd.merge(data3['Person'][['pwtyp', 'psexpfac', 'pwpcl', 'pwaudist','pstyp', 'pspcl', 'psaudist', 'hhno', 'ptpass']],
                            data3['Household'][['hhtaz', 'hhparcel', 'hhno']],
                            on = 'hhno')
    df_summary = pd.DataFrame(columns=['DaysimOutputs', f'{survey_year}Survey', 
                                    f'Difference (DaysimOutputs - {survey_year}Survey)', 
                                    f'% Difference (DaysimOutputs - {survey_year}Survey)'],
                            index=['Total Persons', 'Total Households', 
                                    'Average Household Size', 'Average Trips Per Person', 'Average Trip Length',
                                    'Vehicle Miles per Person', 'Average Distance to Work (Non-Home)', 'Average Distance to School (Non-Home)'])
    df_summary = df_summary.astype('float64')

    trip_ok_1 = data1['Trip'][['travdist', 'trexpfac', 'dorp']].query('travdist > 0 and travdist < 200')
    trip_ok_3 = data3['Trip'][['travdist', 'trexpfac', 'dorp']].query('travdist > 0 and travdist < 200')

    ##Basic Summaries
    #Total Households, Persons, and Trips
    tp1 = data1['Person']['psexpfac'].sum()  # total persons
    tp3 = data3['Person']['psexpfac'].sum()
    th1 = data1['Household']['hhexpfac'].sum()  # total households
    th3 = data3['Household']['hhexpfac'].sum()
    ttr1 = trip_ok_1['trexpfac'].sum()  # total trips
    ttr3 = trip_ok_3['trexpfac'].sum()
    ahhs1 = tp1 / th1  # average household size
    ahhs3 = tp3 / th3
    ntr1 = ttr1 / tp1  # average number of trips per person
    ntr3 = ttr3 / tp3
    atl1 = weighted_average(trip_ok_1, 'travdist', 'trexpfac', grouper=None)  # average trip length
    atl3 = weighted_average(trip_ok_3, 'travdist', 'trexpfac', grouper=None)
    driver_trips1 = trip_ok_1[['dorp', 'travdist', 'trexpfac']].query('dorp == "Driver"')  # vehicle miles (unweighted)
    driver_trips3 = trip_ok_3[['dorp', 'travdist', 'trexpfac']].query('dorp == "Driver"')
    vmpp1sp = (driver_trips1['travdist'].multiply(driver_trips1['trexpfac'])).sum()  # weighted vehicle miles
    vmpp3sp = (driver_trips3['travdist'].multiply(driver_trips3['trexpfac'])).sum()
    vmpp1 = vmpp1sp / tp1  # vehicle miles per person
    vmpp3 = vmpp3sp / tp3

    #Work Location
    wrkrs1 = merge_per_hh_1[['pwtyp', 'hhtaz', 'psexpfac', 'pwpcl', 'pwaudist', 'hhparcel']].\
        query('pwtyp == "Paid Full-Time Worker" or pwtyp == "Paid Part-Time Worker"')
    wrkrs3 = merge_per_hh_3[['pwtyp', 'hhtaz', 'psexpfac', 'pwpcl', 'pwaudist', 'hhparcel']].\
        query('pwtyp == "Paid Full-Time Worker" or pwtyp == "Paid Part-Time Worker"')
    wrkr_1_hzone = pd.merge(wrkrs1, taz_subarea, left_on = 'hhtaz', right_on = 'TAZ')
    wrkr_3_hzone = pd.merge(wrkrs3, taz_subarea, left_on = 'hhtaz', right_on = 'TAZ')
    # only take those in-person workers: usual work location parcel != home location parcel
    workers_1 = wrkr_1_hzone.query('pwpcl != hhparcel and pwaudist > 0 and pwaudist < 200').copy()
    workers_3 = wrkr_3_hzone.query('pwpcl != hhparcel and pwaudist > 0 and pwaudist < 200').copy()
    workers_1['Share (%)'] = workers_1['psexpfac'] / workers_1['psexpfac'].sum()
    workers_3['Share (%)'] = workers_3['psexpfac'] / workers_3['psexpfac'].sum()
    workers1_avg_dist = weighted_average(workers_1, 'pwaudist', 'psexpfac')
    workers3_avg_dist = weighted_average(workers_3, 'pwaudist', 'psexpfac')
    #School Location
    st1 = merge_per_hh_1[['pstyp', 'hhtaz', 'psexpfac', 'pspcl', 'psaudist', 'hhparcel']].\
        query('pstyp == "Full-Time Student" or pstyp == "Part-Time Student"')
    st3 = merge_per_hh_3[['pstyp', 'hhtaz', 'psexpfac', 'pspcl', 'psaudist', 'hhparcel']].\
        query('pstyp == "Full-Time Student" or pstyp == "Part-Time Student"')
    st_1_hzone = pd.merge(st1, taz_subarea, 'outer', left_on = 'hhtaz', right_on = 'TAZ')
    st_3_hzone = pd.merge(st3, taz_subarea, 'outer', left_on = 'hhtaz', right_on = 'TAZ')
    # only take those in-person students: usual school/university location parcel != home location parcel
    students_1 = st_1_hzone.query('pspcl != hhparcel and psaudist > 0 and psaudist < 200').copy()
    students_3 = st_3_hzone.query('pspcl != hhparcel and psaudist > 0 and psaudist < 200').copy()
    students_1['Share (%)'] = students_1['psexpfac'] / students_1['psexpfac'].sum()
    students_3['Share (%)'] = students_3['psexpfac'] / students_3['psexpfac'].sum()
    students1_avg_dist = weighted_average(students_1, 'psaudist', 'psexpfac')
    students3_avg_dist = weighted_average(students_3, 'psaudist', 'psexpfac')

    df_summary.loc[:, ['DaysimOutputs', f'{survey_year}Survey']] = [[tp1, tp3],
                                                                [th1, th3],
                                                                [ahhs1, ahhs3],
                                                                [ntr1, ntr3],
                                                                [atl1, atl3],
                                                                [vmpp1, vmpp3],
                                                                [workers1_avg_dist, workers3_avg_dist],
                                                                [students1_avg_dist, students3_avg_dist]]
    df_summary = get_differences(df_summary, 'DaysimOutputs', f'{survey_year}Survey', 1)

    # table
    display(df_summary.style.format({'DaysimOutputs': '{:.1f}', 
                                    f'{survey_year}Survey': '{:.1f}', 
                                    f'Difference (DaysimOutputs - {survey_year}Survey)': '{:.1f}',
                                    f'% Difference (DaysimOutputs - {survey_year}Survey)': '{:.1f}%'}))

    # figure
    fig = px.bar(
        df_summary.loc[['Average Household Size',
                        'Average Trips Per Person',
                        'Average Trip Length',
                        'Vehicle Miles per Person',
                        'Average Distance to Work (Non-Home)',
                        'Average Distance to School (Non-Home)'], :].reset_index(),
        x='index',
        y=['DaysimOutputs', f'{survey_year}Survey'],
        barmode='group',
        title=f'Daysim Outputs vs Survey Summary ({tag})',
        labels={'index': 'Metric', 'value': 'Value', 'variable': 'Source'},
    )
    fig.update_layout(xaxis_title='', yaxis_title='', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [4]:
hh_and_pop(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Total Persons,4306155.0,4221182.1,84972.9,2.0%
Total Households,1759281.0,1733407.2,25873.8,1.5%
Average Household Size,2.4,2.4,0.0,0.5%
Average Trips Per Person,3.8,3.8,-0.0,-0.8%
Average Trip Length,5.8,6.6,-0.8,-11.5%
Vehicle Miles per Person,16.8,15.4,1.4,9.0%
Average Distance to Work (Non-Home),12.1,11.7,0.4,3.2%
Average Distance to School (Non-Home),5.5,4.2,1.3,31.5%


In [5]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
hh_and_pop(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Total Persons,323575.0,337021.8,-13446.8,-4.0%
Total Households,134808.0,135324.1,-516.1,-0.4%
Average Household Size,2.4,2.5,-0.1,-3.6%
Average Trips Per Person,3.9,4.0,-0.1,-1.3%
Average Trip Length,4.5,6.5,-2.0,-30.1%
Vehicle Miles per Person,13.6,13.9,-0.3,-2.4%
Average Distance to Work (Non-Home),9.5,8.3,1.1,13.6%
Average Distance to School (Non-Home),4.2,4.6,-0.4,-8.1%


## Population by District

In [6]:
data_daysim['Person'] = data_daysim['Person'].merge(data_daysim['Household'][['hhno', 'hhtaz']], on='hhno', how='left')
data_fullsurvey['Person'] = data_fullsurvey['Person'].merge(data_fullsurvey['Household'][['hhno', 'hhtaz']], on='hhno', how='left')
data_daysim['Person'] = get_subarea(_data=data_daysim['Person'], taz_subarea=taz_subarea, taz_colname='hhtaz')
data_fullsurvey['Person'] = get_subarea(_data=data_fullsurvey['Person'], taz_subarea=taz_subarea, taz_colname='hhtaz')

result_daysim = data_daysim['Person'].groupby(by='DistrictFlowName')['psexpfac'].sum()
result_fullsurvey = data_fullsurvey['Person'].groupby(by='DistrictFlowName')['psexpfac'].sum()
result_daysim.loc['Total'] = result_daysim.sum()
result_fullsurvey.loc['Total'] = result_fullsurvey.sum()
# Concatenate the two results into a single DataFrame for comparison
comparison_df = pd.concat(
    [result_daysim.rename('DaysimOutputs'), result_fullsurvey.rename(f'{survey_year}Survey')],
    axis=1
)
comparison_df = comparison_df.loc[district_flow_name.values()]
# table
display(comparison_df.style.format('{:,.0f}'))

,DaysimOutputs,2023Survey
DistrictFlowName,,
Bellevue (excluding downtown),"136,949","148,910"
Bellevue Downtown,"18,107","9,443"
Kirkland,"95,301","75,425"
Redmond,"73,218","103,244"
Seattle (excluding Seattle downtown),"711,443","693,485"
Seattle downtown,"89,865","82,039"
Rest,"3,181,272","3,108,589"


## Workers by District

In [7]:
from collections import OrderedDict

def ppl_by_district(data1=data_daysim, data3=data_fullsurvey, pptyp='worker', pptyp_label='Worker'):
    """
    pptyp: person type that you want to query, 'worker', or 'student'.
    pptyp_label: 'Worker', 'Student'.
    """
    name1 = 'DaysimOutputs'
    name3 = f'{survey_year}Survey'
    #Workers, and students by District
    workers_per_taz_1 = data1['Person'][[f'p{pptyp[0]}taz', 'psexpfac']].groupby(f'p{pptyp[0]}taz').sum()['psexpfac']
    workers_per_taz_3 = data3['Person'][[f'p{pptyp[0]}taz', 'psexpfac']].groupby(f'p{pptyp[0]}taz').sum()['psexpfac']
    workers_per_taz = pd.DataFrame.from_dict(OrderedDict(((f'Number of {pptyp_label}s (' + name1 + ')', workers_per_taz_1), 
                                                            (f'Number of {pptyp_label}s (' + name3 + ')', workers_per_taz_3))))
    workers_per_taz_district = pd.merge(workers_per_taz, taz_subarea, left_index = True, right_on = 'TAZ')
    workers_per_district = workers_per_taz_district[[f'Number of {pptyp_label}s (' + name1 + ')', 
                                                     f'Number of {pptyp_label}s (' + name3 + ')', 'DistrictFlowName']].groupby('DistrictFlowName').sum()
    workers_per_district = get_differences(workers_per_district, f'Number of {pptyp_label}s (' + name1 + ')', 
                                                                 f'Number of {pptyp_label}s (' + name3 + ')', 
                                                                 0) 
    workers_per_district = workers_per_district.loc[district_flow_name.values()]
    display(workers_per_district.style.format({f'Number of {pptyp_label}s (' + name1 + ')': '{:,.0f}',
                                               f'Number of {pptyp_label}s (' + name3 + ')': '{:,.0f}',
                                               f'Difference (Number of {pptyp_label}s ({name1}) - Number of {pptyp_label}s ({name3}))': '{:,.0f}',
                                               f'% Difference (Number of {pptyp_label}s ({name1}) - Number of {pptyp_label}s ({name3}))': '{:,.1f}%', }))

    fig = px.bar(
        workers_per_district.reset_index(),
        x='DistrictFlowName',
        y=[f'Number of {pptyp_label}s (' + name1 + ')', f'Number of {pptyp_label}s (' + name3 + ')'],
        barmode='group',
        title=f'{pptyp_label}s by District',
        labels={'value': f'Number of {pptyp_label}s', 'variable': 'Source', 'DistrictFlowName': 'District'}
    )
    fig.update_layout(xaxis_title='', 
                      yaxis_title=f'Number of {pptyp_label}s', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [8]:
ppl_by_district(data1=data_daysim, data3=data_fullsurvey, pptyp='worker', pptyp_label='Worker')

,Number of Workers (DaysimOutputs),Number of Workers (2023Survey),Difference (Number of Workers (DaysimOutputs) - Number of Workers (2023Survey)),% Difference (Number of Workers (DaysimOutputs) - Number of Workers (2023Survey))
DistrictFlowName,,,,
Bellevue (excluding downtown),"88,456","86,218","2,238",3.0%
Bellevue Downtown,"80,096","33,543","46,553",139.0%
Kirkland,"39,981","58,906","-18,925",-32.0%
Redmond,"97,588","78,797","18,791",24.0%
Seattle (excluding Seattle downtown),"434,385","312,757","121,628",39.0%
Seattle downtown,"252,529","178,833","73,696",41.0%
Rest,"1,343,758","1,035,486","308,272",30.0%


## Students by District

In [9]:
ppl_by_district(data1=data_daysim, data3=data_fullsurvey, pptyp='student', pptyp_label='Student')

,Number of Students (DaysimOutputs),Number of Students (2023Survey),Difference (Number of Students (DaysimOutputs) - Number of Students (2023Survey)),% Difference (Number of Students (DaysimOutputs) - Number of Students (2023Survey))
DistrictFlowName,,,,
Bellevue (excluding downtown),"47,388","60,036","-12,648",-21.0%
Bellevue Downtown,"1,905",300,"1,605",535.0%
Kirkland,"21,178","23,455","-2,277",-10.0%
Redmond,"10,063","12,993","-2,930",-23.0%
Seattle (excluding Seattle downtown),"188,038","177,046","10,992",6.0%
Seattle downtown,"17,374","11,458","5,916",52.0%
Rest,"631,156","644,688","-13,532",-2.0%


## Population by Person Type

In [10]:
def pop_by_pptyp(person_daysim, person_fullsurvey):
    result_daysim = person_daysim.groupby(by='pptyp')['psexpfac'].sum()
    result_fullsurvey = person_fullsurvey.groupby(by='pptyp')['psexpfac'].sum()
    result_daysim.loc['Total'] = result_daysim.sum()
    result_fullsurvey.loc['Total'] = result_fullsurvey.sum()
    # Concatenate the two results into a single DataFrame for comparison
    comparison_df = pd.concat(
        [result_daysim.rename('DaysimOutputs'), result_fullsurvey.rename(f'{base_year}Survey')],
        axis=1
    )

    comparison_df.index.name = 'Person Type'
    # table
    display(comparison_df.loc[ptype_cat.values()].style.format('{:,.0f}'))

In [11]:
pop_by_pptyp(data_daysim['Person'], data_fullsurvey['Person'])

,DaysimOutputs,2024Survey
Person Type,,
Full-Time Worker,"1,746,787","1,771,100"
Part-Time Worker,"420,163","431,954"
Non-Working Adult Age 65+,"417,292","408,446"
Non-Working Adult Age <65,"444,376","564,035"
University Student,"267,869","88,730"
High School Student Age 16+,"109,550","152,255"
Child Age 5-15,"613,249","569,767"
Child Age 0-4,"286,869","234,895"


In [12]:
pop_by_pptyp(data_daysim_bkr['Person'], data_fullsurvey_bkr['Person'])

,DaysimOutputs,2024Survey
Person Type,,
Full-Time Worker,"135,779","149,856"
Part-Time Worker,"30,828","29,516"
Non-Working Adult Age 65+,"30,816","30,256"
Non-Working Adult Age <65,"32,337","39,976"
University Student,"19,056","9,678"
High School Student Age 16+,"7,945","11,954"
Child Age 5-15,"44,707","48,466"
Child Age 0-4,"22,107","17,319"


## Population by Person Type and District

In [13]:
pop_by_type_and_district = data_daysim['Person'].groupby(['pptyp', 'DistrictFlowName'])['psexpfac'].sum().unstack('DistrictFlowName')
pop_by_type_and_district.index.name = 'Person Type'
display(pop_by_type_and_district.loc[ptype_cat.values()].style.format('{:,.0f}'))

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Rest,Seattle (excluding Seattle downtown),Seattle downtown
Person Type,,,,,,,
Full-Time Worker,"54,003","7,993","42,154","31,629","1,239,405","320,840","50,763"
Part-Time Worker,"12,299","1,927","9,445","7,157","301,281","76,472","11,582"
Non-Working Adult Age 65+,"13,313","3,100","8,304","6,099","314,556","63,178","8,742"
Non-Working Adult Age <65,"14,485","1,784","8,909","7,159","350,461","54,142","7,436"
University Student,"8,163","1,222","5,519","4,152","178,237","63,485","7,091"
High School Student Age 16+,"4,131",262,"2,040","1,512","88,285","13,114",206
Child Age 5-15,"21,904","1,087","12,021","9,695","488,073","78,530","1,939"
Child Age 0-4,"8,651",732,"6,909","5,815","220,974","41,682","2,106"


## Transit Pass Ownership

In [14]:
def transit_pass_ownership(data1, data2, data3, tag='PSRC Region'):
    ttp1 = data1['Person']['ptpass'].multiply(data1['Person']['psexpfac']).sum()
    ttp2 = data2['Person'].loc[data2['Person']['ptpass'] > 0, 'psexpfac'].sum()
    ttp3 = data3['Person'].loc[data3['Person']['ptpass'] > 0, 'psexpfac'].sum()
    ppp1 = ttp1 / get_total(data1['Person']['psexpfac'])
    ppp3 = ttp3 / get_total(data3['Person']['psexpfac'])
    tpass = pd.DataFrame(index = ['Total Passes', 'Passes per Person'])
    tpass['DaysimOutputs'] = [ttp1, ppp1]
    tpass[f'{survey_year}Survey'] = [ttp3, ppp3]
    tpass = get_differences(tpass, 'DaysimOutputs', f'{survey_year}Survey', [0, 2])
    # table
    display(tpass.style.format('{:,.1f}'))
    # bar plot
    fig = px.bar(
        tpass.loc[['Total Passes'], ['DaysimOutputs', f'{survey_year}Survey']].reset_index(),
        x='index',
        y=['DaysimOutputs', f'{survey_year}Survey'],
        barmode='group',
        title=f'Total Transit Passes Comparison ({tag})',
        labels={'index': 'Metric', 'value': 'Total Passes', 'variable': 'Source'},
    )
    fig.update_layout(xaxis_title='', 
                      yaxis_title='Total Passes', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [15]:
transit_pass_ownership(data1=data_daysim, data2=data_survey, data3=data_fullsurvey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Total Passes,"996,310.0","1,162,962.6","-166,653.0",-14.3
Passes per Person,0.2,0.3,-0.0,-16.0


In [16]:
transit_pass_ownership(data1=data_daysim_bkr, data2=data_survey_bkr, data3=data_fullsurvey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Total Passes,"94,121.0","99,080.9","-4,960.0",-5.0
Passes per Person,0.3,0.3,-0.0,-1.1


## Automobile Ownership

In [17]:
def auto_ownership(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region'):
    #Auto Ownership
    # only includes stats from data1 (model outputs) and data3 (full survey), as data2 (daysim-formatted) removed many records
    ao1 = 100 * data1['Household'][['hhvehs','hhexpfac']].groupby('hhvehs').sum()['hhexpfac'] / data1['Household']['hhexpfac'].sum()
    veh3_ok = data3['Household'].query('hhvehs >= 0')
    ao3 = 100 * veh3_ok[['hhvehs','hhexpfac']].groupby('hhvehs').sum()['hhexpfac'] / data3['Household']['hhexpfac'].sum()
    ao3 = ao3.reset_index()
    ao3.loc[(ao3['hhvehs']==4), 'hhexpfac'] = ao3.loc[(ao3['hhvehs']>=4), 'hhexpfac'].sum()
    ao3 = ao3[ao3['hhvehs'].isin([0, 1, 2, 3, 4])]
    ao3.set_index('hhvehs', inplace=True)
    for i in range(5, len(ao1)):
        ao1[4] = ao1[4] + ao1[i]
        ao1 = ao1.drop([i])
    for i in range(5, len(ao3)):
        ao3[4] = ao3[4] + ao3[i]
        ao3 = ao3.drop([i])

    ao = pd.DataFrame()

    # read in ACS dataset
    if tag == 'PSRC Region':
        acs_data_ = acs_data
    else:
        acs_data_ = acs_data_bkr
    autos= pd.read_excel(acs_data_,sheet_name = 'AutosTotal', engine = 'openpyxl')
    acs_auto_share = pd.DataFrame(autos['Share'] * 100).dropna(inplace=False)

    ao['Percent of Households (DaysimOutputs)'] = ao1
    ao[f'Percent of Households ({survey_year}Survey)'] = ao3
    ao['Percent of Households (ACS)'] = acs_auto_share 
    ao = get_differences_wt_fullsurvey(ao, 'Percent of Households (DaysimOutputs)', 
                                        f'Percent of Households ({survey_year}Survey)', 
                                        'Percent of Households (ACS)', 1, need_diff_percent=False)
    aonewcol = ['0', '1', '2', '3', '4+']
    ao['Number of Vehicles in Household'] = aonewcol
    ao = ao.reset_index()
    ao = ao.drop(columns = ['hhvehs'])
    ao = ao.set_index('Number of Vehicles in Household')
    # table
    display(ao.style.format('{:,.1f}%'))
    # plot
    fig = px.bar(
        ao.reset_index(),
        x='Number of Vehicles in Household',
        y=['Percent of Households (DaysimOutputs)', 
           f'Percent of Households ({survey_year}Survey)', 
           'Percent of Households (ACS)'],
        barmode='group',
        title=f'Automobile Ownership by Household ({tag})',
        labels={'value': 'Percent of Households', 'variable': 'Source', 'Number of Vehicles in Household': 'Number of Vehicles'}
    )
    fig.update_layout(yaxis_title='Percent of Households', 
                      xaxis_title='Number of Vehicles in Household', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [18]:
auto_ownership(data1=data_daysim, data3=data_fullsurvey, tag='PSRC Region')

,Percent of Households (DaysimOutputs),Percent of Households (2023Survey),Percent of Households (ACS),Difference (Percent of Households (DaysimOutputs) - Percent of Households (2023Survey)),Difference (Percent of Households (DaysimOutputs) - Percent of Households (ACS))
Number of Vehicles in Household,,,,,
0,7.8%,8.3%,8.0%,-0.6%,-0.2%
1,32.6%,33.1%,31.2%,-0.5%,1.4%
2,37.5%,37.6%,35.5%,-0.0%,2.0%
3,15.3%,13.0%,14.8%,2.3%,0.5%
4+,6.8%,7.9%,7.9%,-1.1%,-1.1%


In [19]:
auto_ownership(data1=data_daysim_bkr, data3=data_fullsurvey_bkr, tag='BKR')

,Percent of Households (DaysimOutputs),Percent of Households (2023Survey),Percent of Households (ACS),Difference (Percent of Households (DaysimOutputs) - Percent of Households (2023Survey)),Difference (Percent of Households (DaysimOutputs) - Percent of Households (ACS))
Number of Vehicles in Household,,,,,
0,6.4%,8.3%,8.9%,-1.9%,-2.5%
1,32.0%,38.4%,43.2%,-6.4%,-11.2%
2,39.4%,38.2%,33.6%,1.2%,5.8%
3,15.5%,11.8%,10.5%,3.8%,5.0%
4+,6.6%,3.3%,3.8%,3.4%,2.8%
